In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.fft import dstn, idstn

# ============================================================
# EXPERIMENT 9 – STUDENT EXERCISES
# Roll Number: Not specified in the supplied code
# ============================================================

L = 1.0
U = 1.0
FINAL_TIME = 45.0

ghia_y = np.array([0.0, .1016, .1719, .2813, .4531, .5,
                   .6172, .7344, .8516, .9531, .9688, 1.0])
ghia_u = np.array([0.0, -.06434, -.10150, -.15662, -.21090, -.20581,
                   -.13641, .00332, .23151, .68717, .78871, 1.0])

ghia_x = np.array([0.0, .0625, .0938, .1563, .2266, .5,
                   .8047, .8594, .9063, .9531, .9688, 1.0])
ghia_v = np.array([0.0, .09233, .12317, .16077, .17507, .05454,
                   -.24533, -.22445, -.16914, -.08864, -.05906, 0.0])


def simulate(N, RE, final_time, scheme="upwind"):

    h = L / (N - 1)
    nu = U * L / RE

    dt_target = min(0.30 * h / U, 0.20 * h**2 / nu)
    steps = int(np.ceil(final_time / dt_target))
    dt = final_time / steps

    x = np.linspace(0, L, N)
    y = np.linspace(0, L, N)

    omega = np.zeros((N, N))
    psi = np.zeros_like(omega)

    k = np.arange(1, N - 1)

    eigenvalues = 4.0 / h**2 * (
        np.sin(np.pi * k[:, None] / (2 * (N - 1)))**2
        + np.sin(np.pi * k[None, :] / (2 * (N - 1)))**2
    )

    def solve_streamfunction(w):

        spectral = dstn(w[1:-1, 1:-1], type=1, norm="ortho")

        p = np.zeros_like(w)

        p[1:-1, 1:-1] = idstn(
            spectral / eigenvalues,
            type=1,
            norm="ortho"
        )

        return p

    def wall_vorticity(w, p):

        w[0, 1:-1] = -2.0 * p[1, 1:-1] / h**2

        w[-1, 1:-1] = (
            -2.0 * p[-2, 1:-1] / h**2
            - 2.0 * U / h
        )

        w[1:-1, 0] = -2.0 * p[1:-1, 1] / h**2

        w[1:-1, -1] = -2.0 * p[1:-1, -2] / h**2

    def rhs(w, p):

        c = w[1:-1, 1:-1]

        e = w[1:-1, 2:]
        wst = w[1:-1, :-2]

        n = w[2:, 1:-1]
        s = w[:-2, 1:-1]

        ue = (
            p[2:, 2:] + p[2:, 1:-1]
            - p[:-2, 2:] - p[:-2, 1:-1]
        ) / (4 * h)

        uw = (
            p[2:, 1:-1] + p[2:, :-2]
            - p[:-2, 1:-1] - p[:-2, :-2]
        ) / (4 * h)

        vn = -(
            p[2:, 2:] + p[1:-1, 2:]
            - p[2:, :-2] - p[1:-1, :-2]
        ) / (4 * h)

        vs = -(
            p[1:-1, 2:] + p[:-2, 2:]
            - p[1:-1, :-2] - p[:-2, :-2]
        ) / (4 * h)

        if scheme == "upwind":

            fe = ue * np.where(ue >= 0, c, e)
            fw = uw * np.where(uw >= 0, wst, c)

            fn = vn * np.where(vn >= 0, c, n)
            fs = vs * np.where(vs >= 0, s, c)

        else:

            fe = ue * 0.5 * (c + e)
            fw = uw * 0.5 * (wst + c)

            fn = vn * 0.5 * (c + n)
            fs = vs * 0.5 * (s + c)

        convection = (fe - fw + fn - fs) / h

        diffusion = nu * (
            e + wst + n + s - 4 * c
        ) / h**2

        face_div = np.max(
            np.abs((ue - uw + vn - vs) / h)
        )

        return diffusion - convection, face_div

    max_face_div = 0.0

    for iteration in range(steps):

        psi = solve_streamfunction(omega)

        wall_vorticity(omega, psi)

        rate, face_div = rhs(omega, psi)

        omega[1:-1, 1:-1] += dt * rate

        max_face_div = max(max_face_div, face_div)

        if not np.isfinite(omega).all():
            raise RuntimeError("Numerical solution became unstable")

    psi = solve_streamfunction(omega)

    wall_vorticity(omega, psi)

    u = np.zeros_like(psi)
    v = np.zeros_like(psi)

    u[1:-1, 1:-1] = (
        psi[2:, 1:-1] - psi[:-2, 1:-1]
    ) / (2 * h)

    v[1:-1, 1:-1] = -(
        psi[1:-1, 2:] - psi[1:-1, :-2]
    ) / (2 * h)

    u[-1, :] = U

    rate, final_face_div = rhs(omega, psi)

    uc = np.interp(ghia_y, y, u[:, N // 2])
    vc = np.interp(ghia_x, x, v[N // 2, :])

    rmse_u = np.sqrt(np.mean((uc - ghia_u)**2))
    rmse_v = np.sqrt(np.mean((vc - ghia_v)**2))

    return {
        "N": N,
        "RE": RE,
        "steps": steps,
        "dt": dt,
        "final_time": steps * dt,
        "u": u,
        "v": v,
        "psi": psi,
        "x": x,
        "y": y,
        "u_rmse": rmse_u,
        "v_rmse": rmse_v,
        "u_center": u[N // 2, N // 2],
        "v_center": v[N // 2, N // 2],
        "steady_residual": np.max(np.abs(rate)),
        "max_face_div": max_face_div,
        "final_face_div": final_face_div
    }


# ============================================================
# QUESTION 1 – GRID REFINEMENT
# N = 33, 65 AND 97
# ============================================================

grid_results = []

for N in [33, 65, 97]:

    result = simulate(
        N=N,
        RE=100,
        final_time=45,
        scheme="upwind"
    )

    grid_results.append(result)

print("\nQUESTION 1 – GRID REFINEMENT")

print("N\tSteps\t\tDelta t\t\tU RMSE\t\tV RMSE")

for r in grid_results:

    print(
        f'{r["N"]}\t'
        f'{r["steps"]}\t\t'
        f'{r["dt"]:.6f}\t'
        f'{r["u_rmse"]:.6f}\t'
        f'{r["v_rmse"]:.6f}'
    )

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for r in grid_results:

    N = r["N"]

    axes[1].plot(
        r["u"][:, N // 2],
        r["y"],
        label=f'N = {N}'
    )

    axes[2].plot(
        r["x"],
        r["v"][N // 2, :],
        label=f'N = {N}'
    )

axes[0].contour(
    grid_results[1]["x"],
    grid_results[1]["y"],
    grid_results[1]["psi"],
    levels=22
)

axes[0].set(
    title="Streamlines (N=65)",
    xlabel="x / L",
    ylabel="y / L",
    aspect="equal"
)

axes[1].plot(ghia_u, ghia_y, "ko", label="Ghia")
axes[1].set(
    title="Vertical Centreline",
    xlabel="u / U",
    ylabel="y / L"
)

axes[2].plot(ghia_x, ghia_v, "ko", label="Ghia")
axes[2].set(
    title="Horizontal Centreline",
    xlabel="x / L",
    ylabel="v / U"
)

for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("experiment_9_question_1.png", dpi=160)
plt.show()


# ============================================================
# QUESTION 2 – REYNOLDS NUMBER = 400
# ============================================================

re400 = simulate(
    N=65,
    RE=400,
    final_time=100,
    scheme="upwind"
)

print("\nQUESTION 2 – REYNOLDS NUMBER = 400")

print(f'Reynolds number = {re400["RE"]}')
print(f'Grid size = {re400["N"]} x {re400["N"]}')
print(f'Time steps = {re400["steps"]}')
print(f'Time step = {re400["dt"]:.8f} s')
print(f'Final time = {re400["final_time"]:.4f} s')
print(f'Centre u velocity = {re400["u_center"]:.6f}')
print(f'Centre v velocity = {re400["v_center"]:.6f}')
print(f'Steady residual = {re400["steady_residual"]:.3e}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].contour(
    re400["x"],
    re400["y"],
    re400["psi"],
    levels=25
)

axes[0].set(
    title="Streamlines at Re = 400",
    xlabel="x / L",
    ylabel="y / L",
    aspect="equal"
)

axes[1].plot(
    re400["u"][:, 65 // 2],
    re400["y"],
    label="Re = 400"
)

axes[1].plot(
    ghia_u,
    ghia_y,
    "ko",
    label="Ghia Re = 100"
)

axes[1].set(
    title="Vertical Centreline Velocity",
    xlabel="u / U",
    ylabel="y / L"
)

for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("experiment_9_question_2.png", dpi=160)
plt.show()


# ============================================================
# QUESTION 3 – CENTRED AVERAGE VS DONOR-CELL UPWIND
# ============================================================

upwind = simulate(
    N=65,
    RE=100,
    final_time=45,
    scheme="upwind"
)

centered = simulate(
    N=65,
    RE=100,
    final_time=45,
    scheme="centered"
)

print("\nQUESTION 3 – CENTRED AVERAGE COMPARISON")

print("Scheme\t\t\tU RMSE\t\tV RMSE\t\tCentre U\tCentre V")

print(
    f'{"Donor-cell upwind":<24}'
    f'{upwind["u_rmse"]:.6f}\t\t'
    f'{upwind["v_rmse"]:.6f}\t\t'
    f'{upwind["u_center"]:.6f}\t'
    f'{upwind["v_center"]:.6f}'
)

print(
    f'{"Centred average":<24}'
    f'{centered["u_rmse"]:.6f}\t\t'
    f'{centered["v_rmse"]:.6f}\t\t'
    f'{centered["u_center"]:.6f}\t'
    f'{centered["v_center"]:.6f}'
)

print(
    f'\nUpwind steady residual = '
    f'{upwind["steady_residual"]:.3e}'
)

print(
    f'Centred steady residual = '
    f'{centered["steady_residual"]:.3e}'
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(
    upwind["u"][:, 32],
    upwind["y"],
    label="Donor-cell upwind"
)

axes[0].plot(
    centered["u"][:, 32],
    centered["y"],
    label="Centred average"
)

axes[0].plot(
    ghia_u,
    ghia_y,
    "ko",
    label="Ghia"
)

axes[0].set(
    title="Vertical Centreline",
    xlabel="u / U",
    ylabel="y / L"
)

axes[1].plot(
    upwind["x"],
    upwind["v"][32, :],
    label="Donor-cell upwind"
)

axes[1].plot(
    centered["x"],
    centered["v"][32, :],
    label="Centred average"
)

axes[1].plot(
    ghia_x,
    ghia_v,
    "ko",
    label="Ghia"
)

axes[1].set(
    title="Horizontal Centreline",
    xlabel="x / L",
    ylabel="v / U"
)

for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("experiment_9_question_3.png", dpi=160)
plt.show()

print("\nALL STUDENT EXERCISES COMPLETED")


QUESTION 1 – GRID REFINEMENT
N	Steps		Delta t		U RMSE		V RMSE
33	4800		0.009375	0.005661	0.006486
65	9600		0.004687	0.003674	0.006526
97	20736		0.002170	0.003257	0.006169


/tmp/ipykernel_3812/2185639032.py:277: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()



QUESTION 2 – REYNOLDS NUMBER = 400
Reynolds number = 400
Grid size = 65 x 65
Time steps = 21334
Time step = 0.00468735 s
Final time = 100.0000 s
Centre u velocity = -0.135364
Centre v velocity = 0.071937
Steady residual = 1.838e-11


/tmp/ipykernel_3812/2185639032.py:343: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()



QUESTION 3 – CENTRED AVERAGE COMPARISON
Scheme			U RMSE		V RMSE		Centre U	Centre V
Donor-cell upwind       0.003674		0.006526		-0.205301	0.055471
Centred average         0.001719		0.003747		-0.207532	0.057202

Upwind steady residual = 1.327e-11
Centred steady residual = 4.470e-11

ALL STUDENT EXERCISES COMPLETED
